### PSD Pipeline
#### Table of Contents:
* [1. Environment Setup](#1-environment-setup)
    * [1.1 Library Imports](#11-library-imports)
    * [1.2 Dataset loading/inspection](#12-dataset-loadinginspection)
* [2. Data Segmentation](#data-segmentation)
* [3. Feature Extraction and Class Distribution](#3-feature-extraction--class-distribution)
    * [3.1 PSD](#31-psd-feature-extraction)
    * [3.2 Class Distribution](#32-class-distribution)
* [4. Classification Schemes](#4-classification-schemes)
    * [4.1 Emotional vs. Neutral](#41-emotional-vs-neutral-remap--class-distribution)
    * [4.2 Positive vs. Negative](#42-positive-vs-negative-remap--class-distribution)
* [5. Model Training](#5-model-training)
    * [5.1 XGBoost](#51-xgboost)
        * [Emotional vs. Neutral](#511-emotional-vs-neutral)
        * [Positive vs. Negative](#512-positive-vs-negative)




### 1. Environment Setup

##### 1.1 Library Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy.signal import welch

import matplotlib.pyplot as plt
import seaborn as sns

import optuna
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score, balanced_accuracy_score, classification_report


/Users/connor/miniconda3/envs/ml/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


##### 1.2 Dataset loading/inspection

In [2]:
dataset_path = Path('DEED')
eeg_dataset = []
subject_ids = []

for file in dataset_path.iterdir():
    mat = loadmat(file)
    eeg = mat['Data']
    fname = file.stem  
    label_part = [part for part in fname.split("_") if part.startswith("E")][0]
    subject_part = [part for part in fname.split("_") if part.startswith("S")][0]
    label = int(label_part[1:])  
    subject_id = subject_part[1:-1]  # last two digits = subject number
    
    eeg_dataset.append((eeg, label))
    subject_ids.append(subject_id)

print(f"Loaded {len(eeg_dataset)} trials.")
print(f"Unique subjects: {len(set(subject_ids))}")
print(f"Subject IDs: {sorted(set(subject_ids))}")
print("Example shapes:", [(arr.shape, lbl) for arr, lbl in eeg_dataset[:3]])

print("\nSample filename → label mapping:")
for file, (_, label) in zip(dataset_path.iterdir(), eeg_dataset[:10]):
    print(f"  {file.stem} → E{label}")

Loaded 533 trials.
Unique subjects: 34
Subject IDs: ['002', '003', '004', '005', '007', '011', '012', '013', '014', '015', '016', '017', '018', '020', '021', '022', '023', '024', '025', '027', '028', '029', '030', '031', '032', '033', '034', '035', '036', '037', '038', '039', '040', '042']
Example shapes: [((6, 290000), 2), ((6, 36000), 2), ((6, 51000), 3)]

Sample filename → label mapping:
  G_S0321_M1_E2_R1_N2_raw_ref → E2
  G_S0213_M3_E2_R7_N2_raw_ref → E2
  G_S0393_M2_E3_R5_REM_raw_ref → E3
  G_S0243_M3_E2_R2_N2_raw_ref → E2
  G_S0311_M3_E5_R2_N2_raw_ref → E5
  G_S0031_M1_E3_R4_nan_raw_ref → E3
  G_S0043_M2_E2_R5_N2_raw_ref → E2
  G_S0072_M1_E0_R11_N1_raw_ref → E0
  G_S0242_M1_E3_R3_W_raw_ref → E3
  G_S0342_M2_E3_R3_N2_raw_ref → E3


### 2. Data Segmentation

Done into 20s windows.

In [3]:
def segmentation(eeg_dataset, subject_ids, window_sec, fs):
    window_size = int(window_sec * fs)
    X = []
    y = []
    groups = []

    for (eeg_array, label), sid in zip(eeg_dataset, subject_ids):
        n_samples = eeg_array.shape[1]
        start = 0
        while start + window_size <= n_samples:
            window = eeg_array[:, start:start + window_size]
            X.append(window)
            y.append(label)
            groups.append(sid)
            start += window_size  

    return X, y, groups

windows, window_labels, window_groups = segmentation(eeg_dataset, subject_ids, 20, 200)
print(f"Total 20 second windows: {len(windows)}")
print(f"Unique subjects: {len(set(window_groups))}")

Total 20 second windows: 7490
Unique subjects: 34


### 3. Feature Extraction & Class Distribution

PSD features are extracted from each 20s window using Welch's method. For each channel, log band power 
and relative band power are computed across the five standard frequency bands (delta, theta, alpha, beta, 
gamma), alongside time-domain mean and variance. Windows are then remapped into two binary classification schemes: Emotional vs. Neutral and Positive vs. Negative.

##### 3.1 PSD Feature Extraction

In [4]:
def extract_psd_features(segmented_windows, labels, fs=200):
    freq_bands = {
        'delta': (0.5, 4),
            'theta': (4, 8),
            'alpha': (8, 12),
            'beta': (12, 30),
            'gamma': (30, 45)
    }
    
    windows = np.array(segmented_windows)
    n_windows, n_channels, n_samples = windows.shape
    
    nperseg = min(512, n_samples)  
    noverlap = nperseg // 2
    
    all_features = []
    
    for ch_idx in range(n_channels):
        ch_data = windows[:, ch_idx, :]
        f, Pxx = welch(ch_data, fs=fs, nperseg=nperseg, noverlap=noverlap, axis=1)
        
        # Band power per band
        ch_band_powers = []
        for band_name, (low, high) in freq_bands.items():
            idx = np.logical_and(f >= low, f <= high)
            band_power = np.trapz(Pxx[:, idx], f[idx], axis=1) 
            ch_band_powers.append(band_power)
        ch_band_powers = np.column_stack(ch_band_powers) 

        # Log band power
        log_band_power = np.log(ch_band_powers + 1e-10)
        all_features.append(log_band_power)

        # Relative band power
        total_power = ch_band_powers.sum(axis=1, keepdims=True)
        relative_band_power = ch_band_powers / (total_power + 1e-10)
        all_features.append(relative_band_power)

        # Time-domain features
        ch_mean = np.mean(ch_data, axis=1, keepdims=True)
        ch_var = np.var(ch_data, axis=1, keepdims=True)
        all_features.append(ch_mean)
        all_features.append(ch_var)

        
    
    
    X = np.hstack(all_features)
    y = np.array(labels)
    
    return X, y

##### 3.2 Class Distribution

In [5]:
X, y = extract_psd_features(windows, window_labels)
print(X.shape)
print(y.shape) 

print("\n=== Total Class Distribution===")
unique, counts = np.unique(y, return_counts=True)
for label, count in zip(unique, counts):
    print(f"E{label}: {count} windows ({count/len(y)*100:.1f}%)")

(7490, 72)
(7490,)

=== Total Class Distribution===
E0: 1294 windows (17.3%)
E1: 305 windows (4.1%)
E2: 1069 windows (14.3%)
E3: 2855 windows (38.1%)
E4: 1698 windows (22.7%)
E5: 269 windows (3.6%)


### 4. Classification Schemes

##### 4.1 Emotional vs. Neutral (remap + class distribution)  
Emotional {E1, E2, E4, E5} vs. Neutral Dream {E3} 

In [6]:
def remap_emotional_neutral(y):
    y = np.array(y)
    keep_mask = np.isin(y, [1, 2, 3, 4, 5])  
    new_y = np.zeros(len(y), dtype=int)
    new_y[np.isin(y, [1, 2, 4, 5])] = 1  
    return new_y[keep_mask], keep_mask

y_en, mask_en = remap_emotional_neutral(y)
X_en = X[mask_en]
groups_en = np.array(window_groups)[mask_en]

print(f"\n=== Emotional vs. Neutral Class Distribution ===")
print(f"  Total: {len(y_en)}")
print(f"  Neutral (0): {np.sum(y_en == 0)} ({np.sum(y_en == 0)/len(y_en)*100:.1f}%)")
print(f"  Emotional (1): {np.sum(y_en == 1)} ({np.sum(y_en == 1)/len(y_en)*100:.1f}%)")


=== Emotional vs. Neutral Class Distribution ===
  Total: 6196
  Neutral (0): 2855 (46.1%)
  Emotional (1): 3341 (53.9%)


In [7]:
# Subject Wise Class Distribution (EN)
print("=== Subject Wise Class Distribution (EN) ===")
for subject in sorted(set(groups_en)):
    mask = groups_en == subject
    labels = y_en[mask]
    unique = np.unique(labels)
    print(f"Subject {subject}: {len(labels)} windows, classes: {unique}, counts: {np.bincount(labels)}")

=== Subject Wise Class Distribution (EN) ===
Subject 002: 185 windows, classes: [0 1], counts: [ 18 167]
Subject 003: 400 windows, classes: [0 1], counts: [188 212]
Subject 004: 245 windows, classes: [0 1], counts: [104 141]
Subject 005: 242 windows, classes: [0 1], counts: [155  87]
Subject 007: 511 windows, classes: [0 1], counts: [ 78 433]
Subject 011: 43 windows, classes: [0], counts: [43]
Subject 012: 95 windows, classes: [0 1], counts: [83 12]
Subject 013: 26 windows, classes: [0 1], counts: [12 14]
Subject 014: 56 windows, classes: [0], counts: [56]
Subject 015: 269 windows, classes: [0 1], counts: [122 147]
Subject 016: 213 windows, classes: [0 1], counts: [184  29]
Subject 017: 213 windows, classes: [0 1], counts: [177  36]
Subject 018: 7 windows, classes: [0], counts: [7]
Subject 020: 105 windows, classes: [0 1], counts: [95 10]
Subject 021: 375 windows, classes: [0 1], counts: [162 213]
Subject 022: 207 windows, classes: [0 1], counts: [112  95]
Subject 023: 173 windows, cla

##### 4.2 Positive vs. Negative (remap + class distribution)
Positive {E4, E5} vs. Negative {E1, E2}

In [8]:
def remap_positive_negative(y):
    y = np.array(y)
    keep_mask = np.isin(y, [1, 2, 4, 5])  
    new_y = np.zeros(len(y), dtype=int)
    new_y[np.isin(y, [4, 5])] = 1  
    return new_y[keep_mask], keep_mask

y_pn, mask_pn = remap_positive_negative(y)
X_pn = X[mask_pn]
groups_pn = np.array(window_groups)[mask_pn]

print(f"\n=== Positive vs. Negative Class Distribution ===")
print(f"  Total: {len(y_pn)}")
print(f"  Negative (0): {np.sum(y_pn == 0)} ({np.sum(y_pn == 0)/len(y_pn)*100:.1f}%)")
print(f"  Positive (1): {np.sum(y_pn == 1)} ({np.sum(y_pn == 1)/len(y_pn)*100:.1f}%)")


=== Positive vs. Negative Class Distribution ===
  Total: 3341
  Negative (0): 1374 (41.1%)
  Positive (1): 1967 (58.9%)


In [9]:
# Subject Wise Class Distribution (PN)
print("=== Subject Wise Class Distribution (PN) ===")
for subject in sorted(set(groups_pn)):
    mask = groups_pn == subject
    labels = y_pn[mask]
    unique = np.unique(labels)
    print(f"Subject {subject}: {len(labels)} windows, classes: {unique}, counts: {np.bincount(labels)}")

=== Subject Wise Class Distribution (PN) ===
Subject 002: 167 windows, classes: [0 1], counts: [89 78]
Subject 003: 212 windows, classes: [0 1], counts: [ 65 147]
Subject 004: 141 windows, classes: [0 1], counts: [68 73]
Subject 005: 87 windows, classes: [0 1], counts: [33 54]
Subject 007: 433 windows, classes: [0 1], counts: [206 227]
Subject 012: 12 windows, classes: [0], counts: [12]
Subject 013: 14 windows, classes: [1], counts: [ 0 14]
Subject 015: 147 windows, classes: [0 1], counts: [ 22 125]
Subject 016: 29 windows, classes: [1], counts: [ 0 29]
Subject 017: 36 windows, classes: [0 1], counts: [28  8]
Subject 020: 10 windows, classes: [0], counts: [10]
Subject 021: 213 windows, classes: [0 1], counts: [ 28 185]
Subject 022: 95 windows, classes: [0 1], counts: [46 49]
Subject 023: 100 windows, classes: [0 1], counts: [21 79]
Subject 024: 235 windows, classes: [0 1], counts: [139  96]
Subject 025: 34 windows, classes: [1], counts: [ 0 34]
Subject 027: 26 windows, classes: [0 1], 

### 5. Model Training
For both models and classification schemes, hyperparameters are optimized using Optuna with 5-fold StratifiedGroupKFold on the full dataset. The resulting best parameters are fixed and used for final evaluation with LOSO. For comparison, performance is also assessed using standard 10-fold cross-validation.

##### General Functions for Training

In [10]:
optuna.logging.set_verbosity(optuna.logging.INFO)

def hyperparameter_training(X, y, groups, model_name, classification_scheme):
    print(f"=== Hyperparameter Tuning - {model_name} ({classification_scheme}) ===")

    def objective(trial):
        params = {
            'n_estimators':     trial.suggest_int('n_estimators', 100, 600),
            'max_depth':        trial.suggest_int('max_depth', 3, 8),
            'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'gamma':            trial.suggest_float('gamma', 0, 5),
            'eval_metric':      'logloss',
        }
        cv = StratifiedGroupKFold(n_splits=5)
        scores = []
        for train_idx, val_idx in cv.split(X, y, groups):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            neg = np.sum(y_train == 0)
            pos = np.sum(y_train == 1)
            params['scale_pos_weight'] = neg/pos

            model = XGBClassifier(**params, n_jobs=-1, random_state = 42)
            model.fit(X_train, y_train)
            preds = model.predict(X_val)

            scores.append(f1_score(y_val, preds, average='macro'))

        return np.mean(scores)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials = 50)
    print(f"Best params: {study.best_params}")
    print(f"Best CV F1: {study.best_value:.4f}")


    # Save best params
    best_params = study.best_params
    best_params['random_state'] = 42
    best_params['n_jobs'] = -1
    return best_params

In [11]:
def loso_loop(X, y, groups, params, model_name, classification_scheme):
    # LOSO evaluation with fixed params
    print(f"\n=== LOSO - {model_name} ({classification_scheme}) ===")
    logo = LeaveOneGroupOut()
    accs = []
    f1s = []
    aurocs = []
    bal_accs = []

    for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups)):
        if len(np.unique(y[test_idx])) < 2:
            continue

        subject = groups[test_idx[0]]
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        #Per fold class weighting
        neg = np.sum(y_train == 0)
        pos = np.sum(y_train == 1)

        fold_params = params.copy()
        fold_params['scale_pos_weight'] = neg/pos

        model = XGBClassifier(**fold_params)
        model.fit(X_train, y_train)

        preds = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]

        accs.append(accuracy_score(y_test, preds))
        f1s.append(f1_score(y_test, preds, average='macro'))
        aurocs.append(roc_auc_score(y_test, proba))
        bal_accs.append(balanced_accuracy_score(y_test, preds))
        print(f"Subject {subject} | Balanced Accuracy: {bal_accs[-1]:.4f} | Accuracy: {accs[-1]:.4f} | F1: {f1s[-1]:.4f} | AUROC: {aurocs[-1]:.4f}")

    print(f"Balanced Accuracy:  {np.mean(bal_accs):.4f} ± {np.std(bal_accs):.4f}")
    print(f"Accuracy:  {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"F1:  {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
    print(f"AUROC:  {np.mean(aurocs):.4f} ± {np.std(aurocs):.4f}")


In [12]:
def ten_fold_cv_loop(X, y, params, model_name, classification_scheme):
    print(f"\n=== 10-Fold CV - {model_name} ({classification_scheme}) ===")
    # 10 Fold Cross CV with same tuned params
    cv_10fold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    accs = []
    f1s = []
    bal_accs = []
    aurocs = []

    for train_idx, test_idx in cv_10fold.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        neg = np.sum(y_train == 0)
        pos = np.sum(y_train == 1)
        fold_params = params.copy()
        fold_params['scale_pos_weight'] = neg / pos

        model = XGBClassifier(**fold_params)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]
        # importance = model.feature_importances_ 
        # print(len(importance))

        accs.append(accuracy_score(y_test, preds))
        f1s.append(f1_score(y_test, preds, average='macro'))
        aurocs.append(roc_auc_score(y_test, proba))
        bal_accs.append(balanced_accuracy_score(y_test, preds))

    print(f"Accuracy:          {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"F1:                {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
    print(f"Balanced Accuracy: {np.mean(bal_accs):.4f} ± {np.std(bal_accs):.4f}")
    print(f"AUROC:             {np.mean(aurocs):.4f} ± {np.std(aurocs):.4f}")

#### 5.1 XGBoost

##### 5.1.1 Emotional vs. Neutral

In [13]:
#Hyperparameter tuning
xgb_en_params = hyperparameter_training(X_en, y_en, groups_en, "XGBoost", "Emotional vs. Neutral")

[I 2026-03-15 23:17:10,691] A new study created in memory with name: no-name-0c72cc76-a351-4729-bc74-1a2e2837d3c2


=== Hyperparameter Tuning - XGBoost (Emotional vs. Neutral) ===


[I 2026-03-15 23:17:19,929] Trial 0 finished with value: 0.48385767622685594 and parameters: {'n_estimators': 520, 'max_depth': 6, 'learning_rate': 0.023320251495596828, 'subsample': 0.7360176709715058, 'colsample_bytree': 0.9111654202469937, 'min_child_weight': 9, 'gamma': 0.40165623896342206}. Best is trial 0 with value: 0.48385767622685594.
[I 2026-03-15 23:17:22,235] Trial 1 finished with value: 0.4814438628953754 and parameters: {'n_estimators': 412, 'max_depth': 7, 'learning_rate': 0.20058656993604695, 'subsample': 0.8644059723188124, 'colsample_bytree': 0.831398347472234, 'min_child_weight': 1, 'gamma': 3.539284118820427}. Best is trial 0 with value: 0.48385767622685594.
[I 2026-03-15 23:17:23,141] Trial 2 finished with value: 0.4786689828014031 and parameters: {'n_estimators': 161, 'max_depth': 4, 'learning_rate': 0.20583649599677536, 'subsample': 0.9272756664117572, 'colsample_bytree': 0.8956982793460936, 'min_child_weight': 8, 'gamma': 4.913957255565094}. Best is trial 0 with

Best params: {'n_estimators': 285, 'max_depth': 3, 'learning_rate': 0.29278802587605035, 'subsample': 0.6266504906639246, 'colsample_bytree': 0.942464906876002, 'min_child_weight': 8, 'gamma': 2.270739838516993}
Best CV F1: 0.4980


In [14]:
#LOSO Evaluation
xgb_loso = loso_loop(X_en, y_en, groups_en, xgb_en_params, "XGBoost", "Emotional vs. Neutral")


=== LOSO - XGBoost (Emotional vs. Neutral) ===
Subject 002 | Balanced Accuracy: 0.5246 | Accuracy: 0.5892 | F1: 0.4503 | AUROC: 0.5196
Subject 003 | Balanced Accuracy: 0.4950 | Accuracy: 0.4950 | F1: 0.4945 | AUROC: 0.4836
Subject 004 | Balanced Accuracy: 0.5223 | Accuracy: 0.5184 | F1: 0.5170 | AUROC: 0.5125
Subject 005 | Balanced Accuracy: 0.5305 | Accuracy: 0.5826 | F1: 0.5300 | AUROC: 0.5414
Subject 007 | Balanced Accuracy: 0.5065 | Accuracy: 0.4932 | F1: 0.4301 | AUROC: 0.5221
Subject 012 | Balanced Accuracy: 0.5698 | Accuracy: 0.6842 | F1: 0.5250 | AUROC: 0.7149
Subject 013 | Balanced Accuracy: 0.4107 | Accuracy: 0.3846 | F1: 0.3203 | AUROC: 0.3810
Subject 015 | Balanced Accuracy: 0.4232 | Accuracy: 0.4275 | F1: 0.4233 | AUROC: 0.3784
Subject 016 | Balanced Accuracy: 0.5321 | Accuracy: 0.5681 | F1: 0.4663 | AUROC: 0.5270
Subject 017 | Balanced Accuracy: 0.4661 | Accuracy: 0.3333 | F1: 0.3255 | AUROC: 0.4005
Subject 020 | Balanced Accuracy: 0.5421 | Accuracy: 0.4952 | F1: 0.4095 

In [15]:
#10Fold CV
xgb_10f = ten_fold_cv_loop(X_en, y_en, xgb_en_params, "XGBoost", "Emotional vs. Neutral")


=== 10-Fold CV - XGBoost (Emotional vs. Neutral) ===
Accuracy:          0.6709 ± 0.0219
F1:                0.6686 ± 0.0222
Balanced Accuracy: 0.6686 ± 0.0222
AUROC:             0.7189 ± 0.0172


##### 5.1.2 Positive vs. Negative

In [16]:
#Hyperparameter tuning
xgb_pn_params = hyperparameter_training(X_pn, y_pn, groups_pn, "XGBoost", "Positive vs. Negative")

[I 2026-03-15 23:19:51,483] A new study created in memory with name: no-name-619aa0b5-8786-45d6-9200-3e5416d7f678


=== Hyperparameter Tuning - XGBoost (Positive vs. Negative) ===


[I 2026-03-15 23:19:57,236] Trial 0 finished with value: 0.5217848679219271 and parameters: {'n_estimators': 548, 'max_depth': 5, 'learning_rate': 0.04701884248455885, 'subsample': 0.6886554347132607, 'colsample_bytree': 0.8740274100020649, 'min_child_weight': 2, 'gamma': 1.8229201563607211}. Best is trial 0 with value: 0.5217848679219271.
[I 2026-03-15 23:20:00,503] Trial 1 finished with value: 0.5385138982200194 and parameters: {'n_estimators': 309, 'max_depth': 6, 'learning_rate': 0.05717604690654094, 'subsample': 0.6799071781324606, 'colsample_bytree': 0.7687196133351465, 'min_child_weight': 7, 'gamma': 2.1950273429254272}. Best is trial 1 with value: 0.5385138982200194.
[I 2026-03-15 23:20:03,307] Trial 2 finished with value: 0.531438206838698 and parameters: {'n_estimators': 199, 'max_depth': 6, 'learning_rate': 0.04516927291542221, 'subsample': 0.808420221895716, 'colsample_bytree': 0.6859352085524437, 'min_child_weight': 7, 'gamma': 0.841980954050297}. Best is trial 1 with valu

Best params: {'n_estimators': 507, 'max_depth': 5, 'learning_rate': 0.28386654614456364, 'subsample': 0.8246734132506995, 'colsample_bytree': 0.9822651330567976, 'min_child_weight': 6, 'gamma': 3.2122263562867595}
Best CV F1: 0.5389


In [17]:
#LOSO Evaluation
xgb_loso = loso_loop(X_pn, y_pn, groups_pn, xgb_pn_params, "XGBoost", "Positive vs. Negative")


=== LOSO - XGBoost (Positive vs. Negative) ===
Subject 002 | Balanced Accuracy: 0.6095 | Accuracy: 0.6108 | F1: 0.6094 | AUROC: 0.6537
Subject 003 | Balanced Accuracy: 0.4707 | Accuracy: 0.5755 | F1: 0.4660 | AUROC: 0.3732
Subject 004 | Balanced Accuracy: 0.5191 | Accuracy: 0.5177 | F1: 0.5175 | AUROC: 0.5020
Subject 005 | Balanced Accuracy: 0.4790 | Accuracy: 0.5287 | F1: 0.4743 | AUROC: 0.4540
Subject 007 | Balanced Accuracy: 0.5121 | Accuracy: 0.5127 | F1: 0.5120 | AUROC: 0.5145
Subject 015 | Balanced Accuracy: 0.4227 | Accuracy: 0.6871 | F1: 0.4273 | AUROC: 0.3567
Subject 017 | Balanced Accuracy: 0.6607 | Accuracy: 0.6111 | F1: 0.5786 | AUROC: 0.7634
Subject 021 | Balanced Accuracy: 0.5001 | Accuracy: 0.7371 | F1: 0.4980 | AUROC: 0.5805
Subject 022 | Balanced Accuracy: 0.5559 | Accuracy: 0.5474 | F1: 0.5165 | AUROC: 0.5643
Subject 023 | Balanced Accuracy: 0.4876 | Accuracy: 0.6600 | F1: 0.4876 | AUROC: 0.5961
Subject 024 | Balanced Accuracy: 0.4591 | Accuracy: 0.4383 | F1: 0.4378 

In [18]:
#10Fold CV
xgb_10f = ten_fold_cv_loop(X_pn, y_pn, xgb_pn_params, "XGBoost", "Positive vs. Negative")


=== 10-Fold CV - XGBoost (Positive vs. Negative) ===
Accuracy:          0.6651 ± 0.0275
F1:                0.6598 ± 0.0269
Balanced Accuracy: 0.6644 ± 0.0264
AUROC:             0.7357 ± 0.0250


#### KNN

In [19]:
optuna.logging.set_verbosity(optuna.logging.INFO)

def hyperparameter_training(X, y, groups, model_name, classification_scheme):
    print(f"=== Hyperparameter Tuning - {model_name} ({classification_scheme}) ===")

    def objective(trial):
        params = {
            'n_neighbors': trial.suggest_int('n_neighbors', 1, 30),
            'weights': trial.suggest_categorical('weights', ['uniform', 'distance']),
            'metric': trial.suggest_categorical('metric', ['euclidean', 'manhattan', 'cosine']),
            'leaf_size': trial.suggest_int('leaf_size', 10, 50),
        }
        cv = StratifiedGroupKFold(n_splits=5)
        scores = []
        for train_idx, val_idx in cv.split(X, y, groups):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]
            groups_train = groups[train_idx]

            X_train_norm = X_train.copy()
            for subj in np.unique(groups_train):
                mask = groups_train == subj
                subj_scaler = StandardScaler()
                X_train_norm[mask] = subj_scaler.fit_transform(X_train[mask])

            val_scaler = StandardScaler()
            X_val_norm = val_scaler.fit_transform(X_val)

            model = KNeighborsClassifier(**params)
            model.fit(X_train_norm, y_train)
            preds = model.predict(X_val_norm)
            scores.append(f1_score(y_val, preds, average='macro'))

        return np.mean(scores)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials = 50)
    print(f"Best params: {study.best_params}")
    print(f"Best CV F1: {study.best_value:.4f}")


    # Save best params
    best_params = study.best_params
    return best_params

In [20]:
def loso_loop(X, y, groups, params, model_name, classification_scheme):
    # LOSO evaluation with fixed params
    print(f"\n=== LOSO - {model_name} ({classification_scheme}) ===")
    logo = LeaveOneGroupOut()
    accs = []
    f1s = []
    aurocs = []
    bal_accs = []

    for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups)):
        if len(np.unique(y[test_idx])) < 2:
            continue

        subject = groups[test_idx[0]]
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        groups_train = groups[train_idx]

        X_train_norm = X_train.copy()
        for subj in np.unique(groups_train):
            mask = groups_train == subj
            subj_scaler = StandardScaler()
            X_train_norm[mask] = subj_scaler.fit_transform(X_train[mask])

        test_scaler = StandardScaler()
        X_test_norm = test_scaler.fit_transform(X_test)

        model = KNeighborsClassifier(**params)
        model.fit(X_train_norm, y_train)
        preds = model.predict(X_test_norm)
        proba = model.predict_proba(X_test_norm)[:, 1]

        accs.append(accuracy_score(y_test, preds))
        f1s.append(f1_score(y_test, preds, average='macro'))
        aurocs.append(roc_auc_score(y_test, proba))
        bal_accs.append(balanced_accuracy_score(y_test, preds))
        print(f"Subject {subject} | Balanced Accuracy: {bal_accs[-1]:.4f} | Accuracy: {accs[-1]:.4f} | F1: {f1s[-1]:.4f} | AUROC: {aurocs[-1]:.4f}")

    print(f"Balanced Accuracy:  {np.mean(bal_accs):.4f} ± {np.std(bal_accs):.4f}")
    print(f"Accuracy:  {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"F1:  {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
    print(f"AUROC:  {np.mean(aurocs):.4f} ± {np.std(aurocs):.4f}")


In [21]:
def ten_fold_cv_loop(X, y, groups, params, model_name, classification_scheme):
    print(f"\n=== 10-Fold CV - {model_name} ({classification_scheme}) ===")
    # 10 Fold Cross CV with same tuned params
    cv_10fold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    accs = []
    f1s = []
    bal_accs = []
    aurocs = []

    for train_idx, test_idx in cv_10fold.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        groups_train = groups[train_idx]

        X_train_norm = X_train.copy()
        for subj in np.unique(groups_train):
            mask = groups_train == subj
            subj_scaler = StandardScaler()
            X_train_norm[mask] = subj_scaler.fit_transform(X_train[mask])

        test_scaler = StandardScaler()
        X_test_norm = test_scaler.fit_transform(X_test)

        model = KNeighborsClassifier(**params)
        model.fit(X_train_norm, y_train)
        preds = model.predict(X_test_norm)
        proba = model.predict_proba(X_test_norm)[:, 1]

        accs.append(accuracy_score(y_test, preds))
        f1s.append(f1_score(y_test, preds, average='macro'))
        aurocs.append(roc_auc_score(y_test, proba))
        bal_accs.append(balanced_accuracy_score(y_test, preds))

    print(f"Accuracy:          {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"F1:                {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
    print(f"Balanced Accuracy: {np.mean(bal_accs):.4f} ± {np.std(bal_accs):.4f}")
    print(f"AUROC:             {np.mean(aurocs):.4f} ± {np.std(aurocs):.4f}")

In [22]:
#Hyperparameter tuning
knn_en_params = hyperparameter_training(X_en, y_en, groups_en, "KNN", "Emotional vs. Neutral")

[I 2026-03-15 23:22:36,092] A new study created in memory with name: no-name-678f0517-c452-4674-be83-7f6b526a7844


=== Hyperparameter Tuning - KNN (Emotional vs. Neutral) ===


[I 2026-03-15 23:22:36,917] Trial 0 finished with value: 0.48839491744680086 and parameters: {'n_neighbors': 20, 'weights': 'uniform', 'metric': 'manhattan', 'leaf_size': 39}. Best is trial 0 with value: 0.48839491744680086.
[I 2026-03-15 23:22:37,111] Trial 1 finished with value: 0.5085770076155767 and parameters: {'n_neighbors': 9, 'weights': 'distance', 'metric': 'euclidean', 'leaf_size': 17}. Best is trial 1 with value: 0.5085770076155767.
[I 2026-03-15 23:22:40,470] Trial 2 finished with value: 0.5080876548300223 and parameters: {'n_neighbors': 29, 'weights': 'uniform', 'metric': 'cosine', 'leaf_size': 23}. Best is trial 1 with value: 0.5085770076155767.
[I 2026-03-15 23:22:43,650] Trial 3 finished with value: 0.5048663747340587 and parameters: {'n_neighbors': 17, 'weights': 'distance', 'metric': 'cosine', 'leaf_size': 21}. Best is trial 1 with value: 0.5085770076155767.
[I 2026-03-15 23:22:44,120] Trial 4 finished with value: 0.4774166455309151 and parameters: {'n_neighbors': 21,

Best params: {'n_neighbors': 12, 'weights': 'uniform', 'metric': 'euclidean', 'leaf_size': 13}
Best CV F1: 0.5140


In [23]:
#LOSO Evaluation
knn_en_loso = loso_loop(X_en, y_en, groups_en, knn_en_params, "KNN", "Emotional vs. Neutral")


=== LOSO - KNN (Emotional vs. Neutral) ===
Subject 002 | Balanced Accuracy: 0.5178 | Accuracy: 0.6216 | F1: 0.4610 | AUROC: 0.4436
Subject 003 | Balanced Accuracy: 0.4750 | Accuracy: 0.4850 | F1: 0.4646 | AUROC: 0.4857
Subject 004 | Balanced Accuracy: 0.5291 | Accuracy: 0.5306 | F1: 0.5268 | AUROC: 0.5009
Subject 005 | Balanced Accuracy: 0.5001 | Accuracy: 0.5083 | F1: 0.4937 | AUROC: 0.5081
Subject 007 | Balanced Accuracy: 0.4982 | Accuracy: 0.4168 | F1: 0.3846 | AUROC: 0.4887
Subject 012 | Balanced Accuracy: 0.4679 | Accuracy: 0.5684 | F1: 0.4362 | AUROC: 0.3630
Subject 013 | Balanced Accuracy: 0.6190 | Accuracy: 0.6154 | F1: 0.6154 | AUROC: 0.5893
Subject 015 | Balanced Accuracy: 0.5301 | Accuracy: 0.5390 | F1: 0.5291 | AUROC: 0.5320
Subject 016 | Balanced Accuracy: 0.4714 | Accuracy: 0.4883 | F1: 0.4090 | AUROC: 0.4278
Subject 017 | Balanced Accuracy: 0.5579 | Accuracy: 0.3756 | F1: 0.3700 | AUROC: 0.5604
Subject 020 | Balanced Accuracy: 0.5711 | Accuracy: 0.3048 | F1: 0.2922 | AU

In [24]:
#10Fold CV
knn_en_10f = ten_fold_cv_loop(X_en, y_en, groups_en, knn_en_params, "KNN", "Emotional vs. Neutral")


=== 10-Fold CV - KNN (Emotional vs. Neutral) ===
Accuracy:          0.5228 ± 0.0164
F1:                0.5097 ± 0.0183
Balanced Accuracy: 0.5134 ± 0.0170
AUROC:             0.5225 ± 0.0176


In [25]:
#Hyperparameter tuning
knn_pn_params = hyperparameter_training(X_pn, y_pn, groups_pn, "KNN", "Positive vs. Negative")

[I 2026-03-15 23:24:22,258] A new study created in memory with name: no-name-f9e3bf07-2fc2-4f64-93dd-8657b73d6629


=== Hyperparameter Tuning - KNN (Positive vs. Negative) ===


[I 2026-03-15 23:24:22,457] Trial 0 finished with value: 0.49931926416536143 and parameters: {'n_neighbors': 4, 'weights': 'distance', 'metric': 'manhattan', 'leaf_size': 23}. Best is trial 0 with value: 0.49931926416536143.
[I 2026-03-15 23:24:22,641] Trial 1 finished with value: 0.4788466597142664 and parameters: {'n_neighbors': 21, 'weights': 'uniform', 'metric': 'euclidean', 'leaf_size': 26}. Best is trial 0 with value: 0.49931926416536143.
[I 2026-03-15 23:24:22,823] Trial 2 finished with value: 0.48319544981901463 and parameters: {'n_neighbors': 28, 'weights': 'uniform', 'metric': 'euclidean', 'leaf_size': 17}. Best is trial 0 with value: 0.49931926416536143.
[I 2026-03-15 23:24:22,997] Trial 3 finished with value: 0.4915349295484133 and parameters: {'n_neighbors': 17, 'weights': 'uniform', 'metric': 'euclidean', 'leaf_size': 45}. Best is trial 0 with value: 0.49931926416536143.
[I 2026-03-15 23:24:23,180] Trial 4 finished with value: 0.4689413040833859 and parameters: {'n_neighb

Best params: {'n_neighbors': 8, 'weights': 'uniform', 'metric': 'cosine', 'leaf_size': 13}
Best CV F1: 0.5079


In [26]:
#LOSO Evaluation
knn_pn_loso = loso_loop(X_pn, y_pn, groups_pn, knn_pn_params, "KNN", "Positive vs. Negative")


=== LOSO - KNN (Positive vs. Negative) ===
Subject 002 | Balanced Accuracy: 0.5107 | Accuracy: 0.5030 | F1: 0.4995 | AUROC: 0.5191
Subject 003 | Balanced Accuracy: 0.4677 | Accuracy: 0.5236 | F1: 0.4671 | AUROC: 0.4889
Subject 004 | Balanced Accuracy: 0.4867 | Accuracy: 0.4894 | F1: 0.4850 | AUROC: 0.5209
Subject 005 | Balanced Accuracy: 0.4562 | Accuracy: 0.4713 | F1: 0.4551 | AUROC: 0.4969
Subject 007 | Balanced Accuracy: 0.4765 | Accuracy: 0.4850 | F1: 0.4638 | AUROC: 0.4706
Subject 015 | Balanced Accuracy: 0.5033 | Accuracy: 0.5374 | F1: 0.4486 | AUROC: 0.4458
Subject 017 | Balanced Accuracy: 0.4911 | Accuracy: 0.4167 | F1: 0.4052 | AUROC: 0.3661
Subject 021 | Balanced Accuracy: 0.5138 | Accuracy: 0.4977 | F1: 0.4244 | AUROC: 0.5205
Subject 022 | Balanced Accuracy: 0.4161 | Accuracy: 0.4211 | F1: 0.4042 | AUROC: 0.3822
Subject 023 | Balanced Accuracy: 0.4102 | Accuracy: 0.5100 | F1: 0.4110 | AUROC: 0.4144
Subject 024 | Balanced Accuracy: 0.5335 | Accuracy: 0.5149 | F1: 0.5149 | AU

In [27]:
#10Fold CV
#Most likely worse due to class imbalance
knn_pn_10f = ten_fold_cv_loop(X_pn, y_pn, groups_pn, knn_pn_params, "KNN", "Positive vs. Negative")


=== 10-Fold CV - KNN (Positive vs. Negative) ===
Accuracy:          0.5504 ± 0.0235
F1:                0.5302 ± 0.0227
Balanced Accuracy: 0.5305 ± 0.0228
AUROC:             0.5433 ± 0.0308
